## Persistence

In [1]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


In [2]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.',
   'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke is a play on words that relies on a stereotype about hipsters (a subculture known for valuing trendy, avant-garde, and obscure things before they become mainstream). \n\nHere is how the double meaning works:\n1. **The literal meaning:** To "be cool" means for food to cool down to a safe, edible temperature after being cooked.\n2. **The slang meaning:** To "be cool" means to be popular, trendy, or ahead of the cultural curve. \n\nThe punchline humorously combines these two concepts: the hipster ate the pizza **before it was cool** (literally too hot to eat) just to be able to say he did it before everyone else.',
   'extr

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words that relies on a stereotype about hipsters (a subculture known for valuing trendy, avant-garde, and obscure things before they become mainstream). \n\nHere is how the double meaning works:\n1. **The literal meaning:** To "be cool" means for food to cool down to a safe, edible temperature after being cooked.\n2. **The slang meaning:** To "be cool" means to be popular, trendy, or ahead of the cultural curve. \n\nThe punchline humorously combines these two concepts: the hipster ate the pizza **before it was cool** (literally too hot to eat) just to be able to say he did it before everyone else.'

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words that relies on a stereotype about hipsters (a subculture known for valuing trendy, avant-garde, and obscure things before they become mainstream). \n\nHere is how the double meaning works:\n1. **The literal meaning:** To "be cool" means for food to cool down to a safe, edible temperature after being cooked.\n2. **The slang meaning:** To "be cool" means to be popular, trendy, or ahead of the cultural curve. \n\nThe punchline humorously combines these two concepts: the hipster ate the pizza **before it was cool** (literally too hot to eat) just to be able to say he did it before everyone else.

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'What do you call a fake noodle?\n\nAn **impasta**!',
   'extras': {'signature': 'EmAKXgFpFH0THYmIP6UWOK0aHdwC4akxc5ByWJES+XpDwq0onTvShBc+LX3dTGLl9Vqm11sIKwpVTUW1k/2FLc5rpApwvzBHoH4FWMf551W/hR1z0/1yT3YF2/9XlmQRosM='}}],
 'explanation': [{'type': 'text',
   'text': 'This is a classic pun-based joke that relies on a play on words:\n\n1. **"Fake noodle"**: This sets up the premise of the joke, referring to something that looks like a noodle or is called a noodle, but isn\'t actually one.\n2. **"Impasta"**: This is a portmanteau (a blend of two words) combining:\n   * **"Imposter"**: Someone who pretends to be someone else (a fraud).\n   * **"Pasta"**: The Italian category of food that noodles belong to.\n\nWhen spoken aloud, "impasta" sounds almost identical to "imposter," making it a humorous, lighthearted punchline about a noodle pretending to be the real deal.',
   'extras': {'signature': 'EmAKXgFpFH0T1fDRXLULEd6GUqoO4gZJvbErqD6o

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words that relies on a stereotype about hipsters (a subculture known for valuing trendy, avant-garde, and obscure things before they become mainstream). \n\nHere is how the double meaning works:\n1. **The literal meaning:** To "be cool" means for food to cool down to a safe, edible temperature after being cooked.\n2. **The slang meaning:** To "be cool" means to be popular, trendy, or ahead of the cultural curve. \n\nThe punchline humorously combines these two concepts: the hipster ate the pizza **before it was cool** (literally too hot to eat) just to be able to say he did it before everyone else.'

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words that relies on a stereotype about hipsters (a subculture known for valuing trendy, avant-garde, and obscure things before they become mainstream). \n\nHere is how the double meaning works:\n1. **The literal meaning:** To "be cool" means for food to cool down to a safe, edible temperature after being cooked.\n2. **The slang meaning:** To "be cool" means to be popular, trendy, or ahead of the cultural curve. \n\nThe punchline humorously combines these two concepts: the hipster ate the pizza **before it was cool** (literally too hot to eat) just to be able to say he did it before everyone else.

## Time Travel

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1dee74-36ae-69e0-8000-aac0b8d47334"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1dee74-36ae-69e0-8000-aac0b8d47334'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-12T22:20:27.508773+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dee74-36ac-601e-bfff-3a38822ef668'}}, tasks=(PregelTask(id='0525ec1c-8183-360b-06cc-de68cfc9da6e', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0TYHSfTlqQigLx3Rc8OE6afNST7PPgmUErahR3xf0OzriQgj8X5qcQNRGeF38u4FgsXFVJpYs+2wKVRTM9D5qML3Fhl9rS7RSOIxKKVLKeYWuaelqK2jT0KMo='}}]}),), interrupts=())

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1dee74-36ae-69e0-8000-aac0b8d47334"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.',
   'extras': {'signature': 'EmAKXgFpFH0T4oUnUsZDnLd6CMQoucwFaE++aNv58Kh7pgyJ2gPiQwceJl7Trnn5+/HV15VgShZoAotKdACnu4/k6kVj5UPZOE2OBvVhSMayqrDn5syYqzAmFC58Zfzft1U='}}],
 'explanation': [{'type': 'text',
   'text': 'This is a pun based on a common stereotype about hipsters. \n\nHere is the breakdown of why it\'s funny:\n\n1. **The Stereotype:** Hipsters are often associated with trying to be overly trendy, liking obscure things, and proudly proclaiming that they liked a band, movie, or trend *before* it became popular ("before it was cool"). \n2. **The Literal Meaning:** In the context of food, "cool" refers to the temperature. Freshly baked pizza is hot, and you have to wait for it to "cool down" before you can eat it without burning your mouth. \n3. **The Punchline:** The joke plays on the double meaning of the word "cool." The hipster burn

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0T4oUnUsZDnLd6CMQoucwFaE++aNv58Kh7pgyJ2gPiQwceJl7Trnn5+/HV15VgShZoAotKdACnu4/k6kVj5UPZOE2OBvVhSMayqrDn5syYqzAmFC58Zfzft1U='}}], 'explanation': [{'type': 'text', 'text': 'This is a pun based on a common stereotype about hipsters. \n\nHere is the breakdown of why it\'s funny:\n\n1. **The Stereotype:** Hipsters are often associated with trying to be overly trendy, liking obscure things, and proudly proclaiming that they liked a band, movie, or trend *before* it became popular ("before it was cool"). \n2. **The Literal Meaning:** In the context of food, "cool" refers to the temperature. Freshly baked pizza is hot, and you have to wait for it to "cool down" before you can eat it without burning your mouth. \n3. **The Punchline:** The joke plays on the double meaning of the word "cool." The h

### Updating State

In [18]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1dee74-36ae-69e0-8000-aac0b8d47334", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1dee7f-90c5-6a36-8001-679cbed5aa61'}}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dee7f-90c5-6a36-8001-679cbed5aa61'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-11-12T22:25:32.234243+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dee74-36ae-69e0-8000-aac0b8d47334'}}, tasks=(PregelTask(id='82772bde-43d0-1083-d25c-d1b667c9b8f1', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn his mouth on the pizza?\n\nBecause he ate it before it was cool.', 'extras': {'signature': 'EmAKXgFpFH0T4oUnUsZDnLd6CMQoucwFaE++aNv58Kh7pgyJ2gPiQwceJl7Trnn5+/HV15VgShZoAotKdACnu4/k6kVj5UPZOE2OBvVhSMayqrDn5syYqzAmFC58Zfzft1U='}}], 'explanation': [{'type': 'text', 'text': 'This is

In [21]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1dee7f-90c5-6a36-8001-679cbed5aa61"}})

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': 'Why did the samosa go to therapy?\n\nBecause it had too many mixed feelings bottled up inside, and its outer shell was about to crack under the pressure!',
   'extras': {'signature': 'EmAKXgFpFH0TVdwr89qtFFstMEvw8Fw4FQFDNoIiYXiHHgex1hIqj6sAYW9IPoshjwFRJFOrBotVXxHf1Gi7JIjLREtdCL8wMf9LZ1fXB++ddW6QleksAlA6HL7EpXE5RGw='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a double meaning (pun) that plays on both the physical characteristics of a samosa and human emotional struggles:\n\n1. **"Mixed feelings bottled up inside":** \n   * *In human terms,* this refers to suppressing emotions and not expressing one\'s feelings. \n   * *In food terms,* a samosa is literally filled with a mixture of ingredients (usually spiced potatoes, peas, or meat) packed tightly on the inside.\n\n2. **"Outer shell was about to crack under the pressure":** \n   * *In human terms,* this describes a mental breakdown or burnout caused by

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': [{'type': 'text', 'text': 'Why did the samosa go to therapy?\n\nBecause it had too many mixed feelings bottled up inside, and its outer shell was about to crack under the pressure!', 'extras': {'signature': 'EmAKXgFpFH0TVdwr89qtFFstMEvw8Fw4FQFDNoIiYXiHHgex1hIqj6sAYW9IPoshjwFRJFOrBotVXxHf1Gi7JIjLREtdCL8wMf9LZ1fXB++ddW6QleksAlA6HL7EpXE5RGw='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a double meaning (pun) that plays on both the physical characteristics of a samosa and human emotional struggles:\n\n1. **"Mixed feelings bottled up inside":** \n   * *In human terms,* this refers to suppressing emotions and not expressing one\'s feelings. \n   * *In food terms,* a samosa is literally filled with a mixture of ingredients (usually spiced potatoes, peas, or meat) packed tightly on the inside.\n\n2. **"Outer shell was about to crack under the pressure":** \n   * *In human terms,* this describes a mental breakdown or burnou